In [ ]:
import sys
print(sys.executable)

In [ ]:
#Libraries installation check
import chromadb, sentence_transformers, langchain, transformers, torch

print("✅ chromadb           :", chromadb.__version__)
print("✅ sentence-transformers:", sentence_transformers.__version__)
print("✅ langchain          :", langchain.__version__)
print("✅ transformers       :", transformers.__version__)
print("✅ torch              :", torch.__version__)
print("GPU available        :", torch.cuda.is_available())

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
vces = model.encode(["Apple revenue was $394 billion", "Tesla profit grew"])
print("shape : ",vces.shape)

In [ ]:
#Preprocessing pipeline and Chunking 
#Always run this cell if new pdfs are added

import sys
sys.path.append("..")   # lets the notebook find your project files

from data_loader import run_preprocessing_pipeline

chunks = run_preprocessing_pipeline()

In [ ]:
import json

# Testing a single chunk
sample = chunks[100]

print(f"Chunk ID : {sample['chunk_id']}")
print(f"company : {sample['company']}")
print(f'source : {sample["source"]}')
print(f"page : {sample['page']}")
print(f"char count : {len(sample['text'])}")

print(f"\nText Preview:\n{'-'*40}")
print(sample['text'])

In [ ]:
import pandas as pd

#Breakdown by company

df = pd.DataFrame(chunks)

summary = df.groupby("company").agg(
  total_chunks = ("chunk_id","count"),
  total_pages = ("page","nunique"),
  avg_chunk_length = ("text", lambda x: round(x.str.len().mean()))
).reset_index()

print(summary.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# chunks per company
df.groupby("company").size().plot(
  kind="bar",ax=axes[0],color=["#FF9900","#00A4EF","#E50914","#76B900"]
)

axes[0].set_title("Chunks per Company")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x',rotation=0)

#Chart length distribution
df["text"].str.len().plot(
  kind='hist',bins=40,ax=axes[1],color="#6E6E6E",edgecolor="#000000"
)
axes[1].set_title("Chunk Length Distribution (chars)")
axes[1].set_xlabel("Characters")

plt.tight_layout()
plt.savefig("../evaluation/results/chunk_distribution.png",dpi=150)
plt.show()

In [1]:
# Run this in 01_data_exploration.ipynb
import sys, json
sys.path.append("..")

from pathlib import Path
from vector_rag.indexer      import index_chunks
from vectorless_rag.indexer  import build_bm25_index

# Load already-processed chunks directly from disk
# No need to re-read the PDFs
chunks_path = Path("../data/processed/chunks.json")

with open(chunks_path, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks from chunks.json")
print(f"   Companies: {set(c['company'] for c in chunks)}\n")

# Index into ChromaDB (Vector RAG)
index_chunks(chunks)

# Index into BM25 (Vectorless RAG)
build_bm25_index(chunks)

print("\n🎉 Both indexes rebuilt successfully!")

✅ Loaded 2108 chunks from chunks.json
   Companies: {'NETFLIX', 'MICROSOFT', 'NVIDIA', 'AMAZON'}

Phase 3A : Vector RAG INDEXER (CHROMADB


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


📥 Already indexed: nothing yet
📤 Chunks to embed:  2108



Embedding & Indexing: 100%|██████████| 22/22 [01:25<00:00,  3.87s/it]


✅ ChromaDB index updated.
   Total vectors in index: 2108

   PHASE 3B — VECTORLESS RAG INDEXER (BM25)

🔨 Building BM25 index over 2108 chunks.....
✅ BM25 index built and saved to vectorless_rag\bm25_index.pkl
   Total chunks indexed: 2108


🎉 Both indexes rebuilt successfully!


In [ ]:
import torch
print(torch.__version__)

1.5.9
